In [1]:
# Import necessary ROS2 libraries
import rclpy
from rclpy.node import Node
import time # For potential delays or checks
import sys
import os
import cv2
import json
from ament_index_python.packages import get_package_share_directory

from utils.vision_functions import VisionHandler

# from pm_vision_manager.va_py_modules.vision_pipline_processing import process_image, set_camera_parameters

# try:
#     from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
# except ImportError:
#     # If the direct import fails, try to add the package path
#     # This is a common workaround for development in Jupyter
#     ros2_ws_src_path = os.path.expanduser('~/ros2_ws/src') # Adjust to your workspace src directory
#     my_vision_package_path = os.path.join(ros2_ws_src_path, 'my_vision_package', 'my_vision_package')
#     if os.path.exists(my_vision_package_path):
#         sys.path.insert(0, os.path.dirname(my_vision_package_path))
#         from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
#     else:
#         print(f"Could not find my_vision_package at {my_vision_package_path}. Please check your path or install the package.")
#         raise

# --- ROS2 Node Initialization and Usage ---

# Initialize ROS2 (only call once per process)
if not rclpy.ok():
    rclpy.init()
    print("ROS2 initialized.")
else:
    print("ROS2 already initialized.")

# Create a simple ROS2 node
# This node will be passed to your VisionProcessClass
node = None
try:
    node = Node('jupyter_vision_test_node')
    node.get_logger().info('Jupyter ROS2 node created.')

    vision_handler = VisionHandler(node) 

    vision_functions = vision_handler.return_available_function_names()
    # node.get_logger().info(f"Available vision functions: {vision_functions}")

    process_file_name = "grayscale_threshold.json"

    try:
        result_dict, message = vision_handler.process_image(
            image_name="cube.png",  # Replace with actual image name
            process_filename=process_file_name
        )
    except Exception as e:
        print(f"Error processing image: {e}")

except Exception as e:
    if node:
        node.get_logger().error(f'An error occurred: {e}')
    else:
        print(f'An error occurred before node creation: {e}')
finally:
    # Always destroy the node and shutdown ROS2 when you are done.
    # This is crucial for proper resource management and to avoid errors on re-execution.
    if node:
        node.destroy_node()
        print("ROS2 node destroyed.")
    if rclpy.ok(): # Check if ROS2 is still active before shutting down
        rclpy.shutdown()
        print("ROS2 shutdown.")

print("Jupyter Notebook ROS2 test completed.")



ROS2 initialized.


[INFO] [1753861411.809155377] [jupyter_vision_test_node]: Jupyter ROS2 node created.
[INFO] [1753861411.918663124] [jupyter_vision_test_node]: Execute Vision!
[INFO] [1753861411.919120668] [jupyter_vision_test_node]: Current vision process: grayscale_threshold.json
[INFO] [1753861411.921237878] [jupyter_vision_test_node]: Vision assistant path config loaded!
[INFO] [1753861411.922414492] [jupyter_vision_test_node]: Vision assistant config loaded!
[INFO] [1753861411.926275283] [jupyter_vision_test_node]: Process meta data loaded!
[WARN] [1753861411.955910128] [jupyter_vision_test_node]: Camera exposure_time not defined!
[WARN] [1753861413.000795284] [jupyter_vision_test_node]: Set coax light bool client not online!
[ERROR] [1753861413.004628708] [jupyter_vision_test_node]: Exception occured
[ERROR] [1753861413.006434305] [jupyter_vision_test_node]: CameraExposureTimeError
[ERROR] [1753861413.008420364] [jupyter_vision_test_node]: Camera exposure time service is not available!
[WARN] [17

ROS2 node destroyed.
ROS2 shutdown.
Jupyter Notebook ROS2 test completed.


In [15]:
import os
import json
import cv2
import numpy as np
from typing import Dict
from skimage import restoration, color
from skimage.restoration import estimate_sigma


def AnalyzeImageHistogram(img) -> Dict[str, any]:
    """
    Computes a two‐peak summary of the grayscale histogram.
    Returns: {"peaks": [peak1, peak2], "valley": valley_value}
    """
    hist = cv2.calcHist([img], [0], None, [256], [0,256]).flatten()
    # Find the two highest peaks and the lowest point between them
    peaks = np.argpartition(hist, -2)[-2:]
    peaks = sorted(peaks.tolist())
    valley = int(np.argmin(hist[peaks[0]:peaks[1]]) + peaks[0])

    return {"peaks": peaks, "valley": int(valley)}

def ComputeEdgeDensity(img) -> Dict[str, float]:
    """
    Runs a Canny edge detector and returns the fraction of edge pixels.
    Returns: {"edge_density": float between 0–1}
    """
    edges = cv2.Canny(img, 100, 200)
    print(f"Edge image shape: {edges.shape}, unique values: {np.unique(edges)}")
    density = float(np.count_nonzero(edges)) / edges.size
    return {"edge_density": density}

def EstimateNoiseLevel(img) -> Dict[str, float]:
    """
    Uses a wavelet‐based noise estimation (skimage) to return sigma.
    Returns: {"sigma": estimated_noise_stddev}
    """
    img = img / 255.0
    sigma_est = restoration.estimate_sigma(img, multichannel=False, average_sigmas=True)
    return {"sigma": float(sigma_est)}

def ComputeSaliencyMapSummary(img) -> Dict[str, any]:
    """
    Computes a simple saliency map via spectral residual and returns:
      - num_salient_regions: how many distinct blobs above threshold
      - avg_region_size: in pixels
    Returns: {"num_salient_regions": int, "avg_region_size": float}
    """
    # spectral residual saliency (OpenCV)
    saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, sal_map = saliency.computeSaliency(img)
    sal_map = (sal_map * 255).astype("uint8")
    # threshold to get binary mask of salient areas
    _, mask = cv2.threshold(sal_map, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    # find connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    # ignore background label 0
    region_sizes = stats[1:, cv2.CC_STAT_AREA] if num_labels > 1 else np.array([])
    num_regions = int(len(region_sizes))
    avg_size = float(region_sizes.mean()) if num_regions > 0 else 0.0

    return {"num_salient_regions": num_regions, "avg_region_size": avg_size}

def compute_image_facts(img) -> dict:

    h, w = img.shape[:2]

    # histogram + stats
    hist = cv2.calcHist([img],[0],None,[256],[0,256]).flatten()
    peaks = np.argpartition(hist, -2)[-2:]; peaks = sorted(peaks.tolist())
    valley = int(np.argmin(hist[peaks[0]:peaks[1]]) + peaks[0])
    contrast_stddev = float(img.std())

    # edges
    v = np.median(img)
    low = int(max(0, 0.66*v)); high = int(min(255, 1.33*v))   # simple auto-canny
    edges = cv2.Canny(img, low, high)
    edge_density = float(np.count_nonzero(edges)/edges.size)

    # blur (variance of Laplacian)
    blur_laplacian_var = float(cv2.Laplacian(img, cv2.CV_64F).var())

    # noise estimate (0–1)
    sigma = float(estimate_sigma(img.astype("float32")/255.0,
                                 channel_axis=None, average_sigmas=True))

    # central symmetry + radius guess (very light-weight)
    cy, cx = h//2, w//2
    # radial mean profile (sample 64 rings)
    rs = np.linspace(5, min(h,w)//2-5, 64).astype(int)
    means = []
    for r in rs:
        mask = np.zeros_like(img, np.uint8)
        cv2.circle(mask, (cx,cy), r, 255, 2)
        m = img[mask>0].mean() if np.any(mask) else 0
        means.append(m)
    means = np.array(means, dtype=np.float32)
    # symmetry score: prominence of strongest radial feature
    center_symmetry = float((means.max()-means.min())/max(1.0, means.std()+1e-6))
    # crude radius = position of extremum mapped back to pixels
    center_blob_radius_px = int(rs[int(means.argmax())]) if means.size else 0

    # quick corner count (preview at defaults)
    corners = cv2.goodFeaturesToTrack(img, maxCorners=500, qualityLevel=0.01, minDistance=7)
    corner_count_preview = int(0 if corners is None else corners.shape[0])

    return {
        "image_id": "image_id_placeholder",  # Placeholder, replace with actual image ID if needed
        "size": {"w": int(w), "h": int(h)},
        "contrast_stddev": contrast_stddev,
        "hist_peaks": peaks, "hist_valley": int(valley),
        "edge_density": edge_density,
        "blur_laplacian_var": blur_laplacian_var,
        "noise_sigma": sigma,
        "auto_canny": {"low": low, "high": high},
        "center_symmetry": center_symmetry,
        "center_blob_radius_px": center_blob_radius_px,
        "corner_count_preview": corner_count_preview
    }


image_name = "cube.png"
image_path = os.path.join("/home/match-mover/Pictures/", image_name)

image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

facts = compute_image_facts(image)
print(f"Image Facts for {image_name}:")
for key, value in facts.items():
    print(f"{key}: {value}")

# Example usage of the functions
# histogram_analysis = AnalyzeImageHistogram(image)
# print(f"Histogram Analysis: {histogram_analysis}")
# edge_density = ComputeEdgeDensity(image)
# print(f"Edge Density: {edge_density}")



Image Facts for cube.png:
image_id: image_id_placeholder
size: {'w': 2592, 'h': 1944}
contrast_stddev: 41.05338649894434
hist_peaks: [16, 17]
hist_valley: 16
edge_density: 0.05832285474775187
blur_laplacian_var: 100.3305745575533
noise_sigma: 0.007559765845632778
auto_canny: {'low': 38, 'high': 78}
center_symmetry: 3.0670689727438174
center_blob_radius_px: 50
corner_count_preview: 500


In [ ]:
# Import necessary ROS2 libraries
import rclpy
from rclpy.node import Node
import time # For potential delays or checks
import sys
import os
import cv2

from pm_co_pilot_vision.utils.vision_functions import VisionFunctions

from pm_vision_manager.va_py_modules.vision_pipline_processing import process_image, set_camera_parameters

try:
    from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
except ImportError:
    # If the direct import fails, try to add the package path
    # This is a common workaround for development in Jupyter
    ros2_ws_src_path = os.path.expanduser('~/ros2_ws/src') # Adjust to your workspace src directory
    my_vision_package_path = os.path.join(ros2_ws_src_path, 'my_vision_package', 'my_vision_package')
    if os.path.exists(my_vision_package_path):
        sys.path.insert(0, os.path.dirname(my_vision_package_path))
        from pm_vision_manager.va_py_modules.vision_assistant_class import VisionProcessClass
    else:
        print(f"Could not find my_vision_package at {my_vision_package_path}. Please check your path or install the package.")
        raise

# --- ROS2 Node Initialization and Usage ---

# Initialize ROS2 (only call once per process)
if not rclpy.ok():
    rclpy.init()
    print("ROS2 initialized.")
else:
    print("ROS2 already initialized.")

# Create a simple ROS2 node
# This node will be passed to your VisionProcessClass
node = None
try:
    node = Node('jupyter_vision_test_node')
    node.get_logger().info('Jupyter ROS2 node created.')

    # Define dummy filenames for testing
    process_filename = "test_pipeline.json" # Replace with actual path
    camera_config_filename = "pm_robot_top_cam_1_Unity.yaml" # Replace with actual path
    image_path = "/home/match-mover/pm_Server/01_PM_Zelle/03_PM_DataBase/pm_assembly_database/Vision/vision_image_db/test_pipeline/"
    image_name = "target.png"

    # load image as image:np.ndarray
    image = cv2.imread(f"{image_path}/{image_name}")

    # Instantiate your VisionProcessClass
    vision_processor = VisionProcessClass(
        vision_node=node,
        process_filename=process_filename,
        camera_config_filename=camera_config_filename,
        launch_as_assistant=False, # Set as per your testing needs
        process_UID='JupyterTestVision',
        run_cross_validation=False # Set as per your testing needs
    )

    node.get_logger().info('VisionProcessClass instantiated.')

    vision_processor._load_process_file()

    set_parameter_success = set_camera_parameters(vision_processor.vision_node, 
                                                    vision_processor.image_processing_handler,
                                                    vision_processor.process_pipeline_list)

    if not set_parameter_success:
        vision_processor.image_processing_handler.vision_routine_done = True
        vision_processor.image_processing_handler.visionOK = False
        # error
        node.get_logger().error('Failed to set camera parameters. Exiting...')
        rclpy.shutdown()
        sys.exit(1)
    
    node.get_logger().info('Camera parameters set successfully.')
        
    vision_processor.image_processing_handler.set_image_metatdata(vision_processor.process_db_path,image_name)
    vision_processor.image_processing_handler.set_initial_image(image)

    #display_image = process_image(self, image, self.process_pipeline_list)
    display_image, vision_results = process_image(vision_processor.vision_node, vision_processor.image_processing_handler, vision_processor.process_pipeline_list)
    final_image = vision_processor.image_processing_handler.get_final_image()

    node.get_logger().info(f'vision results: {vision_results}')

    result_dict = vision_processor.save_vision_results()

    node.get_logger().info(f'vision result: {result_dict}')

    # save final image
    image_name_processed = image_name.replace('.png', '_processed.png')
    cv2.imwrite(f"{image_path}/final_{image_name_processed}", final_image)


except Exception as e:
    if node:
        node.get_logger().error(f'An error occurred: {e}')
    else:
        print(f'An error occurred before node creation: {e}')
finally:
    # Always destroy the node and shutdown ROS2 when you are done.
    # This is crucial for proper resource management and to avoid errors on re-execution.
    if node:
        node.destroy_node()
        print("ROS2 node destroyed.")
    if rclpy.ok(): # Check if ROS2 is still active before shutting down
        rclpy.shutdown()
        print("ROS2 shutdown.")

print("Jupyter Notebook ROS2 test completed.")

ROS2 initialized.


[INFO] [1753712081.674133716] [jupyter_vision_test_node]: Jupyter ROS2 node created.
[INFO] [1753712081.733665100] [jupyter_vision_test_node]: Execute Vision!
[INFO] [1753712081.734142761] [jupyter_vision_test_node]: Current vision process: test_pipeline.json
[INFO] [1753712081.735660802] [jupyter_vision_test_node]: Vision assistant path config loaded!
[INFO] [1753712081.736436293] [jupyter_vision_test_node]: Vision assistant config loaded!
[INFO] [1753712081.739583717] [jupyter_vision_test_node]: Process meta data loaded!
[WARN] [1753712081.745036977] [jupyter_vision_test_node]: Camera exposure_time not defined!
[WARN] [1753712082.763140060] [jupyter_vision_test_node]: Set coax light bool client not online!
[ERROR] [1753712082.766385441] [jupyter_vision_test_node]: Exception occured
[ERROR] [1753712082.768132723] [jupyter_vision_test_node]: CameraExposureTimeError
[ERROR] [1753712082.769800236] [jupyter_vision_test_node]: Camera exposure time service is not available!
[WARN] [17537120

Unique values: [  0 255]
Unique values: [  0 255]
ROS2 node destroyed.
ROS2 shutdown.
Jupyter Notebook ROS2 test completed.
